# Intermediate 03 — Incident Response and Recovery

Move a suspicious agent run through evidence admission, verified containment, independently authorized recovery, and safe effect reconciliation. The lab combines an explicit phase model and hash-linked chronology with executable control-plane evidence for identity, revocation, approval, concurrency, provider outcomes, and metrics.

![Agent incident recovery lifecycle](architecture.svg)

The model may classify or propose. Trusted application and control-plane evidence admits the signal, confirms revocation, consumes approval, and reconciles the provider outcome.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
ns = runpy.run_path('03_incident_recovery.py')
ActorContext, DetectionSignal, RevocationReceipt, ProviderResult, EffectState, build_scenario, recover_scenario, issue_approval, evaluate_incident_controls = (ns[name] for name in ('ActorContext','DetectionSignal','RevocationReceipt','ProviderResult','EffectState','build_scenario','recover_scenario','issue_approval','evaluate_incident_controls'))
now = datetime(2026, 9, 17, 12, 0, tzinfo=timezone.utc)
run, responder, planner, approver, signal, revocation, checkpoint, plan = build_scenario(now=now)

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
detected = run.detect(signal, now=now)
contained = run.contain(revocation, responder=responder, now=now)
proposed = run.propose_recovery(plan, checkpoint, planner=planner, now=now)
assert detected and contained and proposed
assert run.phase.value == 'recovery_pending'
[(d.allowed, d.reason, d.phase.value) for d in (detected, contained, proposed)]

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
self_approver = ActorContext(plan.requested_by, 'north', frozenset({'recovery-approver'}))
self_approval = issue_approval(plan, self_approver, approval_id='nb-self', now=now)
denied = run.authorize_recovery(self_approval, checkpoint, now=now, current_policy_version=run.policy_version, current_credential_version=run.credential_version)
assert not denied.allowed and denied.reason == 'approval-independence'
assert run.phase.value == 'recovery_pending'
denied

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
approval = issue_approval(plan, approver, approval_id='nb-valid', now=now)
approved = run.authorize_recovery(approval, checkpoint, now=now, current_policy_version=run.policy_version, current_credential_version=run.credential_version)
assert approved and run.verify_event_chain()
first = run.execute_effect(attempt_id='nb-attempt-1', capability='ticket:write', now=now, provider=lambda _: ProviderResult(EffectState.CONFIRMED, 'provider:T-7'))
duplicate = run.execute_effect(attempt_id='nb-attempt-2', capability='ticket:write', now=now, provider=lambda _: ProviderResult(EffectState.CONFIRMED, 'duplicate'))
assert first.reason == 'provider-confirmed'
assert duplicate.reason == 'duplicate-confirmed' and duplicate.provider_reference is None
assert 'network:external' in run.revoked_capabilities

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
report = evaluate_incident_controls()
assert report.valid_recovery_rate == 1.0
assert report.attack_block_rate == 1.0
assert report.containment_failure_count == 1
assert report.forbidden_effect_count == 0
assert report.duplicate_effect_count == 0
assert report.trace_completeness_rate == 1.0
report

## 6. Exercise a second failure mode

In [ ]:
uncertain, incident_responder, uncertain_plan = recover_scenario(now=now)
unknown = uncertain.execute_effect(attempt_id='nb-unknown', capability='ticket:write', now=now, provider=lambda _: ProviderResult(EffectState.UNKNOWN))
retry = uncertain.execute_effect(attempt_id='nb-retry', capability='ticket:write', now=now, provider=lambda _: ProviderResult(EffectState.CONFIRMED, 'must-not-run'))
assert unknown.reason == 'outcome-unknown'
assert retry.reason == 'reconcile-required' and retry.provider_reference is None
assert uncertain.reconcile_effect(uncertain_plan.operation_id, ProviderResult(EffectState.CONFIRMED, 'provider:T-7'), responder=incident_responder, now=now)
(unknown, retry, uncertain.effects[uncertain_plan.operation_id])

## 7. Signals and containment require trusted evidence

Natural-language urgency is not detector authority, and a revocation request is not proof that containment completed.

In [ ]:
candidate, responder2, _, _, signal2, receipt2, _, _ = build_scenario(now=now)
evil = DetectionSignal('signal-evil', candidate.run_id, candidate.tenant, 'model-output', 'critical', ('claim-1',), now)
assert candidate.detect(evil, now=now).reason == 'untrusted-detector'
assert candidate.detect(signal2, now=now)
partial = RevocationReceipt(receipt2.receipt_id, receipt2.run_id, receipt2.tenant, receipt2.requested, frozenset({'ticket:write'}), frozenset({'network:external'}), receipt2.responder, receipt2.issuer, now)
containment = candidate.contain(partial, responder=responder2, now=now)
assert containment.reason == 'revocation-incomplete'
assert candidate.phase.value == 'detected'
containment

## 8. Approval binds the exact plan

Changing a target after review changes the canonical plan digest; the old or altered receipt cannot authorize the pending plan.

In [ ]:
from dataclasses import replace
plan_run, responder3, planner3, approver3, signal3, receipt3, checkpoint3, plan3 = build_scenario(now=now)
assert plan_run.detect(signal3, now=now)
assert plan_run.contain(receipt3, responder=responder3, now=now)
assert plan_run.propose_recovery(plan3, checkpoint3, planner=planner3, now=now)
altered = replace(plan3, target='ticket:T-8')
altered_approval = issue_approval(altered, approver3, approval_id='nb-altered', now=now)
bound = plan_run.authorize_recovery(altered_approval, checkpoint3, now=now, current_policy_version=plan_run.policy_version, current_credential_version=plan_run.credential_version)
assert bound.reason == 'approval-binding'
bound

## 9. Operation reservation is atomic

Concurrent attempts share one stable operation ID. Only one reaches the provider; the rest observe reserved or confirmed state.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
concurrent_run, _, _ = recover_scenario(now=now)
provider_calls = []
def invoke(index):
    return concurrent_run.execute_effect(attempt_id=f'nb-concurrent-{index}', capability='ticket:write', now=now, provider=lambda _: (provider_calls.append(index), ProviderResult(EffectState.CONFIRMED, 'provider:T-7'))[1])
with ThreadPoolExecutor(max_workers=8) as pool:
    concurrent_results = list(pool.map(invoke, range(8)))
assert len(provider_calls) == 1
assert sum(result.reason == 'provider-confirmed' for result in concurrent_results) == 1
[result.reason for result in concurrent_results]

## 10. Tamper evidence is not protected storage

The local hash chain detects a changed event, but production still needs access control, append-only storage, clock assurance, retention, and independent integrity protection.

In [ ]:
from dataclasses import replace
tampered = list(run.events)
run.events[0] = replace(run.events[0], reason='changed')
assert not run.verify_event_chain()
run.events = tampered
assert run.verify_event_chain()

## 11. OpenTelemetry SDK: bounded lifecycle spans

The companion exports phase, reason, policy, and correlation attributes to memory without recording prompt, evidence body, credential, or hidden reasoning.

In [ ]:
import importlib
otel = importlib.import_module('03_incident_recovery_otel')
otel_run, spans = otel.demo()
assert otel_run.verify_event_chain()
rendered = repr([(span.name, dict(span.attributes)) for span in spans])
assert len(spans) == 6
assert 'credential-v2' not in rendered and 'trace-7' not in rendered
[(span.name, dict(span.attributes)) for span in spans]

## 12. Production replacement

Production replacement: authenticated telemetry and responders, durable incident/case state, verified revocation propagation, encrypted versioned checkpoints, signed and atomically consumed approvals, durable workflow orchestration, provider idempotency and reconciliation, protected append-only evidence, privacy-aware OTLP export, tested rollback and communications, affected-party handling, and explicit owners. A valid alert or approval never proves an external outcome.

## 13. Exercises

1. Add a `quarantined` read-only mode and prove every write/egress capability stays blocked.
2. Add approval revocation before consumption.
3. Persist phase transitions with optimistic concurrency and reject a stale worker update.
4. Implement provider lookup by operation ID for both confirmed and failed unknown outcomes.
5. Add an OpenTelemetry redaction processor and test that a sensitive attribute never reaches the exporter.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.